# 🧠 MS Lesion Segmentation: Training Notebook
### Stage 1: Infrastructure & Data Ingestion

This notebook is designed to handle training on **Google Colab (T4 GPU)** or your **Local Machine** automatically.

## ⚙️ 1.0 Global Configuration & Hardware Switch
This cell detects where the code is running and sets the correct paths automatically.

In [ ]:
import os, torch

# ─── MANUAL OVERRIDE ─────────────────────────────────────────────────────────
ENVIRONMENT = "local"   # Change to "colab" if using Colab
# ─────────────────────────────────────────────────────────────────────────────

if ENVIRONMENT == "colab":
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR       = "/content"
    DRIVE_ROOT     = "/content/drive/MyDrive"
    DATA_ZIP_DIR   = os.path.join(DRIVE_ROOT, "brain_dataset")   # where ZIPs live
    MODEL_SAVE_DIR = os.path.join(DRIVE_ROOT, "MS_Project/models")
    CACHE_DIR      = "/content/p_cache"
else:
    BASE_DIR       = "."
    DATA_ZIP_DIR   = None          # no ZIPs — data already extracted locally
    MODEL_SAVE_DIR = "./models"    # save locally; rclone will sync to Drive
    CACHE_DIR      = "./data/p_cache"

DATA_DIR      = os.path.join(BASE_DIR, "data/raw")
PROCESSED_DIR = os.path.join(BASE_DIR, "data/processed")

for p in [DATA_DIR, PROCESSED_DIR, MODEL_SAVE_DIR, CACHE_DIR]:
    os.makedirs(p, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_WORKERS = min(8, os.cpu_count() // 2)   # use half CPU cores for data loading
print(f"Environment : {ENVIRONMENT}")
print(f"Device      : {device} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")
print(f"CPU workers : {NUM_WORKERS}")

## 🛠 1.1 Environment Setup
Install core medical AI libraries and mount Google Drive if in Colab.

In [ ]:
if ENVIRONMENT == "colab":
    import subprocess
    subprocess.run(["pip", "install", "-q", "monai[all]", "SimpleITK", "nibabel", "tqdm"], check=True)
print("✅ Workspace ready.")

## 📊 1.2 Data Ingestion (Hybrid Strategy)
We pull the ZIP files from your **Google Drive** and extract them to **local storage** for maximum training speed.

In [ ]:
import glob, zipfile, subprocess

if ENVIRONMENT == "colab" and DATA_ZIP_DIR:
    for category in ["MSLesSeg", "Mendeley", "Long-MR-MS"]:
        local_folder = os.path.join(DATA_DIR, category)
        os.makedirs(local_folder, exist_ok=True)
        for zip_path in glob.glob(os.path.join(DATA_ZIP_DIR, category, "*.zip")):
            print(f"Extracting {os.path.basename(zip_path)}...")
            subprocess.run(["unzip", "-q", "-o", zip_path, "-d", local_folder], check=True)
    print("✅ Extraction complete.")
else:
    # Extract any ZIPs placed by Data_Downloader (local mode); no-op if already extracted
    for category in ["MSLesSeg", "Mendeley", "Long-MR-MS"]:
        dest = os.path.join(DATA_DIR, category)
        for zip_path in glob.glob(os.path.join(dest, "*.zip")):
            flag = zip_path.replace(".zip", ".extracted")
            if not os.path.exists(flag):
                print(f"📦 Extracting {os.path.basename(zip_path)}...")
                with zipfile.ZipFile(zip_path, 'r') as z:
                    z.extractall(dest)
                open(flag, 'w').close()
    print(f"✅ Data ready at: {DATA_DIR}")

## 🔍 1.2.1 Directory Mapping (Diagnostic)
Run this cell to see exactly where your FLAIR and Mask files are located inside the subfolders.

In [ ]:
import os

def map_directory_structure(root_dir, max_depth=3):
    print(f"🔍 Mapping structure for: {root_dir}\n")
    for root, dirs, files in os.walk(root_dir):
        depth = root.replace(root_dir, '').count(os.sep)
        if depth <= max_depth:
            indent = '  ' * depth
            print(f"{indent}📂 {os.path.basename(root)}/")
            if files:
                for f in files[:3]:
                    print(f"{indent}  📄 {f}")
                if len(files) > 3:
                    print(f"{indent}  ... ({len(files)-3} more files)")

map_directory_structure(DATA_DIR)

## 📂 1.2.2 Data Discovery & Pairing
This cell pairs MRI scans with their Ground Truth masks for training across all datasets.

In [ ]:
import glob

def create_data_list(raw_dir):
    data_list = []
    
    # 1. Long-MR-MS
    long_path = os.path.join(raw_dir, 'Long-MR-MS')
    if os.path.exists(long_path):
        for p_folder in glob.glob(os.path.join(long_path, 'patient*')):
            gt = glob.glob(os.path.join(p_folder, '*_gt.nii.gz'))
            flairs = glob.glob(os.path.join(p_folder, '*_FLAIRreg.nii.gz'))
            if gt and flairs:
                for f in flairs:
                    data_list.append({'image': f, 'label': gt[0]})

    # 2. Mendeley
    mendeley_path = os.path.join(raw_dir, 'Mendeley')
    if os.path.exists(mendeley_path):
        flair_imgs = [f for f in glob.glob(os.path.join(mendeley_path, '**/*-Flair.nii'), recursive=True) 
                      if "LESIONSEG" not in f.upper()]
        for f in flair_imgs:
            p_id = os.path.basename(f).split('-')[0]
            mask = os.path.join(os.path.dirname(f), f"{p_id}-LesionSeg-Flair.nii")
            if os.path.exists(mask):
                data_list.append({'image': f, 'label': mask})

    # 3. MSLesSeg
    msles_path = os.path.join(raw_dir, 'MSLesSeg')
    if os.path.exists(msles_path):
        all_nii = glob.glob(os.path.join(msles_path, '**/*.nii*'), recursive=True)
        mask_map = {}
        for m in all_nii:
            if "_MASK" in m.upper():
                parts = os.path.basename(m).split('_')
                if len(parts) >= 2: mask_map[f"{parts[0]}_{parts[1]}"] = m
        
        flairs = [f for f in all_nii if "FLAIR" in f.upper() and "_MASK" not in f.upper()]
        for f in flairs:
            parts = os.path.basename(f).split('_')
            if len(parts) >= 2:
                key = f"{parts[0]}_{parts[1]}"
                if key in mask_map: data_list.append({'image': f, 'label': mask_map[key]})
    
    print(f"✅ Total Aligned Pairs: {len(data_list)}")
    return data_list

train_files = create_data_list(DATA_DIR)

## 🚀 1.3 GPU-Accelerated Preprocessing
Standards volumes to 1mm isotropic spacing using MONAI GPU transforms.

In [ ]:
import SimpleITK as sitk
import numpy as np
from monai.transforms import (Compose, LoadImage, EnsureChannelFirst, Orientation, Spacing, ScaleIntensity)

def get_gpu_preprocessor(is_mask=False):
    return Compose([
        LoadImage(image_only=True),
        EnsureChannelFirst(),
        Orientation(axcodes="RAS"),
        Spacing(pixdim=(1.0, 1.0, 1.0), mode="bilinear" if not is_mask else "nearest"),
        ScaleIntensity() if not is_mask else Compose([]),
    ])

def apply_n4_cpu(img_path):
    img = sitk.ReadImage(img_path)
    mask = sitk.OtsuThreshold(img, 0, 1)
    corrector = sitk.N4BiasFieldCorrectionImageFilter()
    corrector.SetMaximumNumberOfIterations([20, 20, 20])
    return corrector.Execute(img, mask)

print("✅ Preprocessor logic loaded.")

## ⚡ 1.3.1 Bulk Parallel GPU Preprocessing
Standardizes all patient pairs into the `/processed` folder.

In [ ]:
from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm.notebook import tqdm
import nibabel as nib, numpy as np, SimpleITK as sitk, torch
from monai.transforms import Compose, EnsureChannelFirst, Orientation, Spacing, ScaleIntensity

GPU_BATCH = 6       # volumes processed on GPU at once; lower if OOM
N_PROC    = max(1, os.cpu_count() - 2)

# ── Phase A: CPU worker — N4 only, returns numpy arrays ─────────────────────
def run_n4(args):
    idx, pair = args
    try:
        raw  = sitk.ReadImage(pair['image'])
        mask = sitk.OtsuThreshold(raw, 0, 1)
        cor  = sitk.N4BiasFieldCorrectionImageFilter()
        cor.SetMaximumNumberOfIterations([20, 20, 20])
        corrected = cor.Execute(raw, mask)
        img_arr = sitk.GetArrayFromImage(corrected).astype(np.float32)

        lbl_raw = sitk.ReadImage(pair['label'])
        lbl_arr = sitk.GetArrayFromImage(lbl_raw).astype(np.float32)

        spacing = list(reversed(raw.GetSpacing()))   # SimpleITK z,y,x → MONAI x,y,z
        return idx, img_arr, lbl_arr, spacing, pair
    except Exception as e:
        return idx, None, None, None, str(e)

# ── Phase B: GPU batch transform ────────────────────────────────────────────
img_gpu_tf = Compose([EnsureChannelFirst(), Orientation(axcodes="RAS"),
                       Spacing(pixdim=(1.0, 1.0, 1.0), mode="bilinear"), ScaleIntensity()])
lbl_gpu_tf = Compose([EnsureChannelFirst(), Orientation(axcodes="RAS"),
                       Spacing(pixdim=(1.0, 1.0, 1.0), mode="nearest")])

def gpu_process_batch(batch_items):
    """batch_items: list of (idx, img_arr, lbl_arr, spacing, pair)"""
    results = []
    for idx, img_arr, lbl_arr, spacing, pair in batch_items:
        subj_dir = os.path.join(PROCESSED_DIR, f"sub-{idx:03d}")
        img_out  = os.path.join(subj_dir, "image.nii.gz")
        lbl_out  = os.path.join(subj_dir, "label.nii.gz")

        if os.path.exists(img_out) and os.path.exists(lbl_out):
            results.append({'image': img_out, 'label': lbl_out})
            continue
        os.makedirs(subj_dir, exist_ok=True)

        img_t = img_gpu_tf(torch.tensor(img_arr).unsqueeze(0).to(device)).cpu()
        lbl_t = lbl_gpu_tf(torch.tensor(lbl_arr).unsqueeze(0).to(device)).cpu()

        nib.save(nib.Nifti1Image(img_t.numpy().squeeze(), np.eye(4)), img_out)
        nib.save(nib.Nifti1Image(lbl_t.numpy().squeeze(), np.eye(4)), lbl_out)
        results.append({'image': img_out, 'label': lbl_out})
    return results

# ── Orchestrate: submit N4 jobs, process GPU as results come back ────────────
processed_train_files = []
pending_batch = []

print(f"🚀 Two-phase preprocessing: {N_PROC} CPU workers + GPU batches of {GPU_BATCH}")
with ProcessPoolExecutor(max_workers=N_PROC) as pool:
    futures = {pool.submit(run_n4, (i, p)): i for i, p in enumerate(train_files)}
    for fut in tqdm(as_completed(futures), total=len(futures), desc="N4 + GPU pipeline"):
        idx, img_arr, lbl_arr, spacing, pair_or_err = fut.result()
        if img_arr is None:
            print(f"ERROR sub-{idx:03d}: {pair_or_err}")
            continue
        pending_batch.append((idx, img_arr, lbl_arr, spacing, pair_or_err))
        if len(pending_batch) >= GPU_BATCH:
            processed_train_files.extend(gpu_process_batch(pending_batch))
            pending_batch = []

# flush remaining
if pending_batch:
    processed_train_files.extend(gpu_process_batch(pending_batch))

print(f"✅ Done: {len(processed_train_files)} pairs ready.")

## 🧪 1.4 Architecture: 3D Residual U-Net

In [ ]:
from monai.networks.nets import UNet

def get_model(in_channels=1, out_channels=1):
    return UNet(
        spatial_dims=3, in_channels=in_channels, out_channels=out_channels,
        channels=(16, 32, 64, 128, 256), strides=(2, 2, 2, 2),
        num_res_units=2, norm="INSTANCE", act="LEAKYRELU", dropout=0.1
    )

print("✅ Architecture loaded.")

## 📉 1.5 Loss & Metrics

In [ ]:
from monai.losses import DiceLoss
import torch.nn as nn
from scipy.ndimage import label

class MSLesionLoss(nn.Module):
    def __init__(self, alpha=1.0, weight=10.0):
        super(MSLesionLoss, self).__init__()
        self.alpha, self.dice_loss = alpha, DiceLoss(sigmoid=True, squared_pred=True)
        self.bce_loss = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([weight]).to(device))
    def forward(self, input, target):
        return self.dice_loss(input, target) + self.alpha * self.bce_loss(input, target)

print("✅ Loss logic ready.")

## 🚀 1.6 The Hardware-Agnostic Training Engine

In [ ]:
from monai.data import PersistentDataset, DataLoader, pad_list_data_collate
from monai.transforms import (Compose, LoadImaged, EnsureChannelFirstd, SpatialPadd,
                               RandCropByPosNegLabeld, RandFlipd, RandRotate90d,
                               NormalizeIntensityd, ToTensord)
from monai.inferers import sliding_window_inference
from monai.metrics import DiceMetric
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
import time

# ─── Hyperparameters ───────────────────────────────────────────────────────────
EPOCHS        = 100
BATCH_SIZE    = 2       # 2 volumes × 4 crops = 8 patches/step; fits 12GB VRAM
PATCH_SIZE    = (96, 96, 96)
LR            = 1e-4
GRAD_ACCUM    = 2       # effective batch = BATCH_SIZE × GRAD_ACCUM = 4
SAVE_EVERY    = 5       # periodic checkpoint interval (epochs)
# ──────────────────────────────────────────────────────────────────────────────

def train_model(data_list, val_fraction=0.15):
    # Train/Val split
    n_val = max(1, int(len(data_list) * val_fraction))
    train_data, val_data = data_list[n_val:], data_list[:n_val]

    train_tf = Compose([
        LoadImaged(keys=["image", "label"]),
        EnsureChannelFirstd(keys=["image", "label"]),
        NormalizeIntensityd(keys=["image"], nonzero=True),
        SpatialPadd(keys=["image", "label"], spatial_size=PATCH_SIZE),
        RandCropByPosNegLabeld(keys=["image", "label"], label_key="label",
                                spatial_size=PATCH_SIZE, pos=1, neg=1, num_samples=4),
        RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=0),
        RandRotate90d(keys=["image", "label"], prob=0.5),
        ToTensord(keys=["image", "label"])
    ])
    val_tf = Compose([
        LoadImaged(keys=["image", "label"]),
        EnsureChannelFirstd(keys=["image", "label"]),
        NormalizeIntensityd(keys=["image"], nonzero=True),
        ToTensord(keys=["image", "label"])
    ])

    train_ds = PersistentDataset(data=train_data, transform=train_tf,
                                  cache_dir=os.path.join(CACHE_DIR, "train"))
    val_ds   = PersistentDataset(data=val_data,   transform=val_tf,
                                  cache_dir=os.path.join(CACHE_DIR, "val"))

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=(device.type == "cuda"),
                              persistent_workers=(NUM_WORKERS > 0),
                              prefetch_factor=2 if NUM_WORKERS > 0 else None,
                              collate_fn=pad_list_data_collate)
    val_loader   = DataLoader(val_ds, batch_size=1, shuffle=False,
                              num_workers=max(1, NUM_WORKERS // 2),
                              pin_memory=(device.type == "cuda"),
                              collate_fn=pad_list_data_collate)

    model       = get_model().to(device)
    loss_fn     = MSLesionLoss().to(device)
    optimizer   = AdamW(model.parameters(), lr=LR, weight_decay=1e-5)
    scheduler   = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
    scaler      = torch.amp.GradScaler('cuda') if device.type == "cuda" else None
    dice_metric = DiceMetric(include_background=False, reduction="mean")

    best_dice = 0.0
    best_path = os.path.join(MODEL_SAVE_DIR, "best_model.pth")
    ckpt_path = os.path.join(MODEL_SAVE_DIR, "checkpoint_latest.pth")

    for epoch in range(EPOCHS):
        model.train()
        epoch_loss, t0 = 0.0, time.time()
        optimizer.zero_grad()

        for step, batch in enumerate(tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [train]")):
            images = batch["image"].to(device, non_blocking=True)
            labels = batch["label"].to(device, non_blocking=True)

            if scaler:
                with torch.amp.autocast('cuda'):
                    loss = loss_fn(model(images), labels) / GRAD_ACCUM
                scaler.scale(loss).backward()
                if (step + 1) % GRAD_ACCUM == 0:
                    scaler.step(optimizer); scaler.update(); optimizer.zero_grad()
            else:
                loss = loss_fn(model(images), labels) / GRAD_ACCUM
                loss.backward()
                if (step + 1) % GRAD_ACCUM == 0:
                    optimizer.step(); optimizer.zero_grad()
            epoch_loss += loss.item() * GRAD_ACCUM

        scheduler.step()

        # ── Validation (sliding window on full volumes) ────────────────────
        model.eval()
        with torch.no_grad():
            for val_batch in tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [val]", leave=False):
                val_img = val_batch["image"].to(device)
                val_lbl = val_batch["label"].to(device)
                pred = sliding_window_inference(val_img, PATCH_SIZE, 4, model)
                dice_metric(y_pred=(pred.sigmoid() > 0.5).float(), y=val_lbl)
        val_dice = dice_metric.aggregate().item()
        dice_metric.reset()

        elapsed = (time.time() - t0) / 60
        print(f"Epoch {epoch+1:3d} | Loss: {epoch_loss/len(train_loader):.4f} | Val Dice: {val_dice:.4f} | {elapsed:.1f}m")

        # ── Save checkpoints ───────────────────────────────────────────────
        torch.save({'epoch': epoch, 'model': model.state_dict(),
                    'optimizer': optimizer.state_dict(), 'best_dice': best_dice}, ckpt_path)
        if val_dice > best_dice:
            best_dice = val_dice
            torch.save(model.state_dict(), best_path)
            print(f"  ✅ New best Dice: {best_dice:.4f} → saved to {best_path}")
        elif (epoch + 1) % SAVE_EVERY == 0:
            ep_path = os.path.join(MODEL_SAVE_DIR, f"model_epoch{epoch+1}.pth")
            torch.save(model.state_dict(), ep_path)

    print(f"\n🏁 Training complete. Best Val Dice: {best_dice:.4f}")
    return model

print("✅ Training engine ready.")
# train_model(processed_train_files)

In [ ]:
import subprocess

RCLONE_REMOTE     = "gdrive"              # rclone remote name (configured once)
DRIVE_MODELS_PATH = "MS_Project/models"   # path inside your Drive

def sync_models_to_drive():
    src = os.path.abspath(MODEL_SAVE_DIR)
    dst = f"{RCLONE_REMOTE}:{DRIVE_MODELS_PATH}"
    print(f"📤 Syncing {src} → {dst} ...")
    result = subprocess.run(["rclone", "copy", src, dst, "--progress"])
    if result.returncode == 0:
        print("✅ Sync complete — model is now in your Drive.")
    else:
        print("❌ rclone sync failed. Check rclone is installed and configured.")

if ENVIRONMENT == "local":
    sync_models_to_drive()

## ☁️ 1.7 rclone Sync to Google Drive

Syncs `./models/` to Drive after training. Requires a one-time setup:

```bash
# 1. Download rclone (no install needed — just a binary)
#    Windows: https://rclone.org/downloads/
#    Linux:   curl https://rclone.org/install.sh | sudo bash

# 2. Configure once on home machine (opens browser for Google sign-in)
rclone config
# → New remote → name: "gdrive" → type: Google Drive → follow prompts

# 3. Copy config to the university machine (no browser needed there)
#    Windows: C:\Users\<you>\AppData\Roaming\rclone\rclone.conf
#    Linux:   ~/.config/rclone/rclone.conf
```